In [1]:
def flatten_dataset(dataset):
    data_df = pd.DataFrame(dataset)
    answers = pd.json_normalize(data_df['answers'])
    df_answers = pd.DataFrame()
    data_df['answer_text'] = None
    df_answers['pos_answer'] = None
    data_df['ra'] = data_df['ra'].astype(int)
    for c in range(len(answers.columns)):
        answer = pd.json_normalize(answers[c])
        df_answers = pd.concat([df_answers, answer['atext']], axis=1)
        df_answers.rename(columns={'atext': c + 1}, inplace=True)
    for i in range(data_df.shape[0]):
        col = data_df.loc[i, 'ra']
        data_df.loc[i, 'answer_text'] = df_answers.iloc[i, col]
    df_answers['pos_answer'] = df_answers.apply(lambda x: '\n '.join(x.dropna().astype(str)), axis=1)
    data_df = pd.concat([data_df, df_answers], axis=1)
    return data_df

In [2]:
!pip install -q accelerate==0.34.2 peft==0.12.0 bitsandbytes==0.43.3 transformers==4.43.1 trl==0.10.1 sacrebleu evaluate rouge_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.4 MB/s eta 0:00:0

In [3]:
import sys

IN_COLAB = 'google.colab' in sys.modules
RUN_TRAINING_CELLS = IN_COLAB

EXPERIMENT_NAME = 'HeadQA-LLaMA-3_1-8B-v2/'
DRIVE_FOLDER_LOCATION = '/content/drive/MyDrive/MIAR - TFM/LLMs-Experiments/' + EXPERIMENT_NAME

In [4]:
# Mounting google drive
if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
# Using my own Google Drive during the experiment to save all checkpoints and training logs.

if IN_COLAB:
    # Adapted from:  https://robertbrucecarter.com/writing/2020/06/setting-your-working-directory-to-google-drive-in-a-colab-notebook/
    import os

    def create_and_set_working_directory(path: str):
        # check if your project folder exists. if not, it will be created.
        if os.path.isdir(path) == False:
            os.mkdir(path)
            print(path + ' did not exist but was created.')

        # change the OS to use your project folder as the working directory
        os.chdir(path)

        print('Working directory changed to: \n' + path)

    create_and_set_working_directory(DRIVE_FOLDER_LOCATION)
    !pwd

Working directory changed to: 
/content/drive/MyDrive/MIAR - TFM/LLMs-Experiments/HeadQA-LLaMA-3_1-8B-v2/
/content/drive/MyDrive/MIAR - TFM/LLMs-Experiments/HeadQA-LLaMA-3_1-8B-v2


In [6]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
    EarlyStoppingCallback
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
import pandas as pd
import numpy as np
import evaluate

In [7]:
from google.colab import userdata
sec_key = userdata.get('HF_TOKEN')

In [8]:
!huggingface-cli login --token $sec_key

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [9]:
# The model that you want to train from the Hugging Face hub
model_name = "NousResearch/Meta-Llama-3.1-8B-Instruct"

In [10]:
# The instruction dataset to use
dataset_name = "head_qa"

In [11]:
SEP_TOKEN = ''
MASKING_CHANCE = 0.3 #30% chance to replace the answer with '[MASK]'

In [12]:
class QGDataset(Dataset):

    def __init__(
        self,
        data: pd.DataFrame,
        tokenizer: AutoTokenizer,
        source_max_token_len: int,
        target_max_token_len: int
        ):

        self.tokenizer = tokenizer
        self.data = data
        self.source_max_token_len = source_max_token_len
        self.target_max_token_len = target_max_token_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index: int):
        data_row = self.data.iloc[index]

        if np.random.rand() > MASKING_CHANCE:
            answer = data_row['answer_text']
        else:
            answer = '[MASK]'

        source_encoding = tokenizer(
            '{} {} {}'.format(answer, SEP_TOKEN, data_row['pos_answer']),
            max_length= self.source_max_token_len,
            padding='max_length',
            truncation= True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
            )

        target_encoding = tokenizer(
            '{} {} {}'.format(data_row['answer_text'], SEP_TOKEN, data_row['qtext']),
            max_length=self.target_max_token_len,
            padding='max_length',
            truncation = True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
            )

        labels = target_encoding['input_ids']
        labels[labels == 0] = -100

        return dict(
            answer_text = data_row['answer_text'],
            pos_answer = data_row['pos_answer'],
            question = data_row['qtext'],
            input_ids = source_encoding['input_ids'].flatten(),
            attention_mask = source_encoding['attention_mask'].flatten(),
            labels=labels.flatten()
            )

In [13]:
# Reload model in FP16 and merge it with LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, "./results/checkpoint-3000/")
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [14]:
pipeline = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

In [15]:
messages = [
    {
        "role": "user",
        "content": """A partir de ahora eres un chatbot que habla español
        y vas a generar preguntas con cinco opciones de respuesta sobre el tema que
        el usuario ingrese. Utiliza el siguiente ejemplo para generar las preguntas
        Tema:
            Insulina.
        Ejemplo de pregunta:
        Aumenta la lipogénesis:
        a) Insulina.
        b) Glucagón.
        c) Cortisol.
        d) Somatotropina.
        e) Serotonina.
"""
    }
]

In [16]:
outputs = pipeline(
    messages,
    max_new_tokens=300,
)
print(outputs[0]["generated_text"][-1].get('content'))

Tema: Insulina.
  Aumenta la lipogénesis:
  a) Insulina.
  b) Glucagón.
  c) Cortisol.
  d) Somatotropina.
  e) Serotonina.


In [17]:
messages.append({"role": "assistant", "content": outputs[0]["generated_text"][-1].get('content')})

In [18]:
# The instruction dataset to use
dataset_name = "head_qa"
dataset_test = load_dataset(dataset_name, split="test", trust_remote_code=True)

head_qa.py:   0%|          | 0.00/6.21k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

head-qa-es-en-pdfs.zip:   0%|          | 0.00/79.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2657 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2742 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1366 [00:00<?, ? examples/s]

In [19]:
data_test_df = flatten_dataset(dataset_test)

In [20]:
max_seq_length = 1024
test_dataset = QGDataset(
    data=data_test_df,
    tokenizer=tokenizer,
    source_max_token_len=max_seq_length,
    target_max_token_len=max_seq_length
)

In [21]:
# rand_int_arr = np.random.randint(low=0, high=len(test_dataset), size=10)
rand_int_arr = [2496, 1708,  191, 1478, 1184, 1133, 1640,  947, 2587, 1838]
rand_int_arr

[2496, 1708, 191, 1478, 1184, 1133, 1640, 947, 2587, 1838]

In [22]:
for i in rand_int_arr:
    messages.append({"role": "user",
                     "content":"Tema: " + test_dataset[i]["answer_text"]})
    outputs = pipeline(
    messages,
    max_new_tokens=300)
    messages.append({"role": "assistant", "content": outputs[0]["generated_text"][-1].get('content')})
    print(outputs[0]["generated_text"][-1].get('content'))

Tema: Músculo orbicular de los ojos (orbicularis oculi).
 Se innerva mediante un nervio:
 a) Facial.
 b) Oculomotor.
 c) Troclear.
 d) Abducens.
 e) Oftálmico.
Tema: Es la presión de la sangre medida a nivel de la unión de la vena cava con la aurícula derecha.
 Se mide mediante el catéter:
 a) Pulmonar Arterial.
 b) Cava Inferior.
 c) Cava Media.
 d) Cava Superior.
 e) Corazón.
Tema: Asociaciones intermoleculares de 3 hélices extendidas.
 Son ácidos nucleicos:
 a) ADN.
 b) ARN.
 c) ARNmi.
 d) ARNt.
 e) ARNr.
Tema: Inhibidor de la cinasa γ de κB (IKKγ).
 Se encuentra en los linfocitos B:
 a) CD4+.
 b) CD8+.
 c) Linfocitos NK.
 d) Linfocitos T.
 e) Linfocitos B.
Tema: Como un conductor iónico.
 Se encuentra en el sistema nervioso:
 a) Ganglios de la raíz.
 b) Ganglios espinales.
 c) Ganglios basales.
 d) Ganglios de la médula espinal.
 e) Ganglios de la retina.
Tema: Se manifiesta especialmente en tareas cognitivas complejas.
 Se debe a una disminución del flujo sanguíneo cerebral:
 a) H

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Tema: Corindón impurificado con pequeñas cantidades de Cr(III).
 Se utiliza como diurético:
 a) Furosemida.
 b) Etacrinona.
 c) Hidroclorotiazida.
 d) Amilorida.
 e) Digoxina.
Tema: Omeprazol.
 Se utiliza para tratar la hiperplasia prostática:
 a) Finasterida.
 b) Dutasterida.
 c) Tamsulosina.
 d) Omeprazol.
 e) Alendronato.


In [23]:
for i in rand_int_arr:
    print("Pregunta ", i)
    print(test_dataset[i]["question"])
    print(test_dataset[i]["pos_answer"])
    print("Respuesta:\n", test_dataset[i]["answer_text"])

Pregunta  2496
Una sonrisa verdadera, en comparación con una falsa, se caracteriza por la activación anatómica del:
Músculo orbicular de los ojos (orbicularis oculi).
 Músculo orbicular de los labios (orbicularis oris).
 Músculo cigomático mayor (zygomaticus major).
 Músculo risorio (risorius).
Respuesta:
 Músculo orbicular de los ojos (orbicularis oculi).
Pregunta  1708
El término Presión Venosa Central hace referencia a:
La carga o volumen que distiende el ventrículo izquierdo antes de la contracción o sístole.
 Es la presión de la sangre medida a nivel de la unión de la vena cava con la aurícula derecha.
 La carga o volumen que distiende la aurícula izquierda antes de la relajación o diástole.
 La carga o volumen que distiende la aurícula derecha antes de la relajación o diástole.
Respuesta:
 Es la presión de la sangre medida a nivel de la unión de la vena cava con la aurícula derecha.
Pregunta  191
La estructura básica del colágeno está constituida por:
Una elevada proporción de al

In [25]:
import json
with open('generated_questions.json', 'w') as f:
    json.dump(outputs, f)

In [26]:
gen_text = """
Tema: Músculo orbicular de los ojos (orbicularis oculi).
 Se innerva mediante un nervio:
 a) Facial.
 b) Oculomotor.
 c) Troclear.
 d) Abducens.
 e) Oftálmico.
Tema: Es la presión de la sangre medida a nivel de la unión de la vena cava con la aurícula derecha.
 Se mide mediante el catéter:
 a) Pulmonar Arterial.
 b) Cava Inferior.
 c) Cava Media.
 d) Cava Superior.
 e) Corazón.
Tema: Asociaciones intermoleculares de 3 hélices extendidas.
 Son ácidos nucleicos:
 a) ADN.
 b) ARN.
 c) ARNmi.
 d) ARNt.
 e) ARNr.
Tema: Inhibidor de la cinasa γ de κB (IKKγ).
 Se encuentra en los linfocitos B:
 a) CD4+.
 b) CD8+.
 c) Linfocitos NK.
 d) Linfocitos T.
 e) Linfocitos B.
Tema: Como un conductor iónico.
 Se encuentra en el sistema nervioso:
 a) Ganglios de la raíz.
 b) Ganglios espinales.
 c) Ganglios basales.
 d) Ganglios de la médula espinal.
 e) Ganglios de la retina.
Tema: Se manifiesta especialmente en tareas cognitivas complejas.
 Se debe a una disminución del flujo sanguíneo cerebral:
 a) Hipertensión arterial.
 b) Diabetes Mellitus.
 c) Enfermedad de Alzheimer.
 d) Enfermedad de Parkinson.
 e) Isquemia cerebral.
Tema: Vigilar que no se interrumpa la administración de oxígeno al paciente durante el choque.
 Se encarga de la administración de:
 a) Antibióticos.
 b) Analgésicos.
 c) Anticoagulantes.
 d) Vasopresores.
 e) Oxígeno.
Tema: Sixteen Personality Factor Questionnaire, 16PF.
 Se utiliza para evaluar la personalidad de las personas con el objetivo de:
 a) Evaluar la inteligencia.
 b) Evaluar el coeficiente intelectual.
 c) Evaluar la personalidad.
 d) Evaluar el estrés.
 e) Evaluar la motivación.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Tema: Corindón impurificado con pequeñas cantidades de Cr(III).
 Se utiliza como diurético:
 a) Furosemida.
 b) Etacrinona.
 c) Hidroclorotiazida.
 d) Amilorida.
 e) Digoxina.
Tema: Omeprazol.
 Se utiliza para tratar la hiperplasia prostática:
 a) Finasterida.
 b) Dutasterida.
 c) Tamsulosina.
 d) Omeprazol.
 e) Alendronato.
"""

In [27]:
ref_text = """
Una sonrisa verdadera, en comparación con una falsa, se caracteriza por la activación anatómica del:
Músculo orbicular de los ojos (orbicularis oculi).
 Músculo orbicular de los labios (orbicularis oris).
 Músculo cigomático mayor (zygomaticus major).
 Músculo risorio (risorius).

El término Presión Venosa Central hace referencia a:
La carga o volumen que distiende el ventrículo izquierdo antes de la contracción o sístole.
 Es la presión de la sangre medida a nivel de la unión de la vena cava con la aurícula derecha.
 La carga o volumen que distiende la aurícula izquierda antes de la relajación o diástole.
 La carga o volumen que distiende la aurícula derecha antes de la relajación o diástole.

La estructura básica del colágeno está constituida por:
Una elevada proporción de alfa-hélice.
 Una estructura secundaria rica en lámina beta.
 Una estructura primaria rica en aminoácidos aromáticos.
 Asociaciones intermoleculares de 3 hélices extendidas.

En relación con la displasia ectodérmica anhidrótica con inmunodeficiencia (DEA-ID), ¿qué gen de la vía del factor nuclear –κB (NF-κB) está mutado?:
Inhibidor de la cinasa α de κB (IKKα).
 Inhibidor de la cinasa β de κB (IKKβ).
 Inhibidor de la cinasa γ de κB (IKKγ).
 Cinasa 4 asociada al receptor de la interleucina-1 (IRAK-4).

¿Cómo se comporta la disolución durante una electrolisis?:
Como un conductor electrónico.
 Como un conductor iónico.
 Como un semiconductor.
 Como un medio aislante.

A diferencia de la inteligencia fluida, la inteligencia cristalizada:
Implica fundamentalmente aptitudes de inducción y deducción más que de comprensión verbal o riqueza de vocabulario.
 Responde a un concepto mecánico de la inteligencia.
 Evoluciona en forma de U invertida a lo largo del ciclo vital.
 Se manifiesta especialmente en tareas cognitivas complejas.

¿Cuál de los siguientes cuidados de enfermería relacionados con la cardioversión es INCORRECTO?:
Previo al procedimiento se confirmará la persistencia de la arritmia a tratar, mediante un registro del electrocardiograma (ECG) de 12 derivaciones, o bien con una tira de ritmo.
 Monitorizar al paciente con el monitordesfibrilador, seleccionando la derivación electrocardiográfica que muestre la onda R de mayor amplitud (mayor voltaje), que permita detectar correctamente al desfibrilador de forma SINC (sincronizado).
 Vigilar que no se interrumpa la administración de oxígeno al paciente durante el choque.
 Posterior a la cardioversión, en el caso de existir quemaduras cutáneas, se aplicará sulfadiazina de plata sobre éstas.

La evaluación de la personalidad se ha hecho desde diversos modelos y teorías. Desde las teorías factoriales, el test más representativo que ha inspirado el desarrollo de otros es:
Minnesota Multiphasic Personality Inventory, MMPI.
 Sixteen Personality Factor Questionnaire, 16PF.
 Gordon Personal Profile, GPP.
 Tennessee Self Concept Scale, TSCS.

¿Qué es el rubí, piedra preciosa usada en el “láser de rubí”?:
Circón impurificado con pequeñas cantidades de Fe(III).
 Circón impurificado con pequeñas cantidades de Cr(III).
 Corindón impurificado con pequeñas cantidades de Fe(III).
 Corindón impurificado con pequeñas cantidades de Cr(III).

¿Cuál de los siguientes fármacos es un profármaco?:
Vigabatrina.
 Metotrexato.
 Atracurio.
 Omeprazol.
"""

In [28]:
rouge = evaluate.load('rouge')
results_rouge = rouge.compute(predictions=[gen_text],
                         references=[ref_text],
                        use_aggregator=True)
print(list(results_rouge.keys()))
print(results_rouge["rouge1"])
print(results_rouge["rouge2"])
print(results_rouge["rougeL"])

['rouge1', 'rouge2', 'rougeL', 'rougeLsum']
0.35480225988700564
0.2015855039637599
0.26892655367231644


In [29]:
sacrebleu = evaluate.load("sacrebleu")
results_sacrebleu = sacrebleu.compute(predictions=[gen_text],
                                      references=[ref_text])
print(list(results_sacrebleu.keys()))
print(round(results_sacrebleu["score"], 1))

['score', 'counts', 'totals', 'precisions', 'bp', 'sys_len', 'ref_len']
16.9


In [30]:
meteor = evaluate.load('meteor')
results = meteor.compute(predictions=[gen_text], references=[ref_text])
print(round(results['meteor'], 2))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


0.22
